# Magnetic Ridge Set 1 — from coil field to XY ridge-peak dataset

This notebook is the **upstream** companion to `Magnetic_Ridge_Set_2.ipynb`. Here we build the workflow **through** a dense set of magnetic ridge **peak** points sampled on the $z=0$ plane (and lifted to the ridge surface in $z$), suitable for later parameterization.

**Pipeline (high level)**

1. **Poincaré diagnostics** — Trace field lines from $(r\cos\phi, r\sin\phi, 0)$ and record crossings of the fixed-$\phi$ plane. Visually identify radial intervals where the section stays confined versus where trajectories explore large volumes or numerical integration becomes unreliable.
2. **Sparse ridge peaks along lines** — Using the same field-line integration as `Magnetic_Ridge.jl`, detect ridge points where $\mathbf{B}^{\mathsf T}(\nabla|\mathbf{B}|)\mathbf{B}$ changes sign along $\mathbf{b}$, and collect a first cloud of ridge peaks.
3. **Dense XY refinement** — Following the spirit of `magnetic_ridge_3.jl`, treat that sparse cloud as an XY footprint: build a local $z$-interval from neighbors at each grid $(x,y)$, then use **ForwardDiff** for $d|\mathbf{B}|/ds$ along $\mathbf{b}$ and root-find in $z$ to populate a denser ridge dataset.

Ranges in steps 2–3 are **not** auto-inferred: you choose them using the figures from step 1.

### Repository root

Julia `include` paths are relative to the working directory. Set `REPO` to this package root so `Coil.jl` and the coil JSON resolve correctly.

In [10]:
const REPO = raw"C:/Users/danie/Stellarator_Isodrasticity"
cd(REPO)
pwd()

using Pkg
Pkg.activate("c:/Users/danie/Stellarator_Isodrasticity")

  Activating project at `c:\Users\danie\Stellarator_Isodrasticity`


### Imports and `Coil.jl`

Load standard numerical stack plus `Coil.jl`, which provides Biot–Savart evaluation (`eval_B`), coil deserialization (`get_coils_from_file`), and norm helpers used throughout.

In [11]:
using OrdinaryDiffEq
using GLMakie
using StaticArrays
using LinearAlgebra
using ForwardDiff
using Roots
using JLD2

include(joinpath(REPO, "Coil.jl"))

torus_mesh! (generic function with 1 method)

### Load coils

Point `COIL_FILE` at your SIMSOPT-export JSON (same convention as `Trace_line.jl` / `Magnetic_Ridge.jl`). Adjust `Nquad` if you change quadrature resolution.

In [12]:
const COIL_FILE = joinpath(REPO, "landreman_paul.json")
const NQUAD = 64

coils = get_coils_from_file(Float64, COIL_FILE, NQUAD)
length(coils)

16

### Poincaré section diagnostics (Trace_line-style)

For each trial $\phi$, we sweep **initial** radii $r$ at $z=0$ and record each time the orbit crosses the half-plane attached to that $\phi$ (same condition as `Trace_line.jl`: $-x\sin\phi + y\cos\phi = 0$ with appropriate crossing direction handled by the integrator callback).

**How to read the plot:** For a confined magnetic surface, successive crossings typically fill a narrow closed curve in $(R,Z)$ (with $R=\sqrt{x^2+y^2}$ in 3D Cartesian coordinates). If $r$ is too large, you may see sparse scatter, drift, or numerical blow-up—exclude those radii from later stages.

Edit `PHI_TRIALS`, `R_SWEEPS`, and integration length `L_POINCARE` below, then run the cell.

In [14]:
# --- User-adjustable exploration parameters -----------------------------------
const L_POINCARE = 1500.0   # Field-line length parameter (same order as Trace_line.jl)

# Several φ values (rad); add or remove as needed
const PHI_TRIALS = [0.0, π / 5, 2π / 5, 3π / 5, 4π / 5]

# One broad radial sweep per φ to visualize confinement vs loss
const R_SWEEPS = range(0.55, 1.55; length = 25)

# ODE: tangent field direction
function b_field_ode(u, p, t)
    B = eval_B(p.coils, u)
    n = norm(B)
    n < eps(Float64) * 1e6 && return SA[0.0, 0.0, 0.0]
    B / n
end

"""
    poincare_hits_for_r(coils, r, phi, L_max)

Trace one field line and return arrays of (x,y,z) each time the trajectory crosses the φ plane.
"""
function poincare_hits_for_r(coils, r, phi, L_max)
    px, py, pz = Float64[], Float64[], Float64[]
    function plane_condition(u, t, integrator)
        -u[1] * sin(phi) + u[2] * cos(phi)
    end
    function save_hit!(integrator)
        push!(px, integrator.u[1])
        push!(py, integrator.u[2])
        push!(pz, integrator.u[3])
    end
    cb = ContinuousCallback(plane_condition, save_hit!, nothing; save_positions = (false, false))
    u0 = SA[r * cos(phi), r * sin(phi), 0.0]
    prob = ODEProblem(b_field_ode, u0, (0.0, L_max), (coils = coils,))
    sol = solve(
        prob,
        Vern9();
        callback = cb,
        save_everystep = false,
        save_start = false,
        save_end = false,
        reltol = 1e-10,
        abstol = 1e-10,
    )
    return px, py, pz, sol.retcode
end

# Collect per-(φ,r) Poincaré data for plotting
poincare_results = Dict{NTuple{2,Float64},NamedTuple{(:R, :Z, :retcode),Tuple{Vector{Float64},Vector{Float64},Any}}}()

for phi in PHI_TRIALS
    for r in R_SWEEPS
        px, py, pz, rc = poincare_hits_for_r(coils, r, phi, L_POINCARE)
        Rcyl = hypot.(px, py)
        poincare_results[(phi, r)] = (; R = Rcyl, Z = pz, retcode = rc)
    end
end

nphi = length(PHI_TRIALS)
fig = Figure(size = (320 * nphi, 420))
for (k, phi) in enumerate(PHI_TRIALS)
    ax = Axis(
        fig[1, k];
        title = "Poincaré φ = $(round(phi/π, digits=3)) π",
        xlabel = "R = √(x²+y²) (m)",
        ylabel = "Z (m)",
        aspect = DataAspect(),
    )
    for r in R_SWEEPS
        row = poincare_results[(phi, r)]
        isempty(row.R) && continue
        # Color by initial minor radius r so you can correlate curves with labels
        scatter!(ax, row.R, row.Z; markersize = 3, color = r, colormap = :turbo, colorrange = extrema(R_SWEEPS))
    end
    Colorbar(fig[2, k]; limits = extrema(R_SWEEPS), label = "initial r (m)", vertical = false, flipaxis = false)
end
fig

# Optional: quick summary of failed integrations
bad = [(phi, r, poincare_results[(phi, r)].retcode) for phi in PHI_TRIALS for r in R_SWEEPS if poincare_results[(phi, r)].retcode != :Success]
println("Non-success solves (φ, r, retcode): ", length(bad), " entries")
isempty(bad) || display(first(bad, min(5, length(bad))))

5-element Vector{Tuple{Float64, Float64, SciMLBase.ReturnCode.T}}:
 (0.0, 0.55, SciMLBase.ReturnCode.Success)
 (0.0, 0.5916666666666667, SciMLBase.ReturnCode.Success)
 (0.0, 0.6333333333333333, SciMLBase.ReturnCode.Success)
 (0.0, 0.675, SciMLBase.ReturnCode.Success)
 (0.0, 0.7166666666666667, SciMLBase.ReturnCode.Success)

Non-success solves (φ, r, retcode): 125 entries


### Manual selection of tracing ranges

After inspecting the Poincaré grids above, **edit** the structured list below: one radial interval per azimuthal launch angle $\phi$. These tuples drive the sparse ridge collection in the next section.

Example pattern (replace with your own numbers):

- `φ = 0` → `r ∈ [1.15, 1.30]`
- `φ = π/5` → `r ∈ [1.10, 1.25]`
- …

In [15]:
# Each entry: φ => (rmin, rmax, nradial)
const RIDGE_SPARSE_CONFIG = [
    0.0      => (1.15, 1.30, 10),
    π / 5    => (1.10, 1.25, 10),
    2π / 5   => (0.65, 1.20, 10),
    3π / 5   => (0.625, 1.20, 10),
    4π / 5   => (1.10, 1.25, 10),
]

const L_RIDGE_TRACE = 20_000.0  # Field-line length for ridge detection (Magnetic_Ridge.jl scale)

RIDGE_SPARSE_CONFIG

5-element Vector{Pair{Float64, Tuple{Float64, Float64, Int64}}}:
                0.0 => (1.15, 1.3, 10)
 0.6283185307179586 => (1.1, 1.25, 10)
 1.2566370614359172 => (0.65, 1.2, 10)
 1.8849555921538759 => (0.625, 1.2, 10)
 2.5132741228718345 => (1.1, 1.25, 10)

### Ridge callbacks and sparse peak extraction (`Magnetic_Ridge.jl` logic)

We combine:

- **Poincaré crossings** (same half-plane as above) to organize samples along the orbit;
- **Ridge callback** that fires when $\mathbf{B}^{\mathsf T}(\nabla|\mathbf{B}|)\mathbf{B}$ changes sign along the field—implemented with ForwardDiff on an allocation-free Biot–Savart loop (`Magnetic_Ridge.jl`).

Run this cell once to define `create_poincare_cb`, `create_ridge_cb`, and `run_poincare_ridge`.

In [16]:
function create_poincare_cb(phi, px, py, pz)
    function condition(u, t, integrator)
        return -u[1] * sin(phi) + u[2] * cos(phi)
    end
    function affect!(integrator)
        push!(px, integrator.u[1])
        push!(py, integrator.u[2])
        push!(pz, integrator.u[3])
    end
    return ContinuousCallback(condition, affect!)
end

function create_ridge_cb(coils, rx, ry, rz)
    function ridge_condition(u, t, integrator)
        B = eval_B(coils, u)
        function B_agnostic(x)
            B_acc = StaticArrays.zeros(eltype(x), 3)
            for c in coils
                for (ri, Idri) in zip(c.rs, c.Idrs)
                    delta_r = ri - x
                    dist = three_norm(delta_r)
                    B_acc += cross(delta_r, Idri) / (dist^3)
                end
            end
            return B_acc
        end
        J = ForwardDiff.jacobian(B_agnostic, u)
        return dot(B, J, B)
    end
    function save_peak!(integrator)
        push!(rx, integrator.u[1])
        push!(ry, integrator.u[2])
        push!(rz, integrator.u[3])
    end
    return ContinuousCallback(ridge_condition, nothing, save_peak!)
end

function run_poincare_ridge(coils, r, phi, L_max)
    x = r * cos(phi)
    y = r * sin(phi)
    u0 = SA[x, y, 0.0]
    px, py, pz = Float64[], Float64[], Float64[]
    rx, ry, rz = Float64[], Float64[], Float64[]
    cb_p = create_poincare_cb(phi, px, py, pz)
    cb_r = create_ridge_cb(coils, rx, ry, rz)
    cb_total = CallbackSet(cb_p, cb_r)
    prob = ODEProblem(b_field_ode, u0, (0.0, L_max), (coils = coils,))
    solve(prob, Vern9(); callback = cb_total)
    return px, py, pz, rx, ry, rz
end

run_poincare_ridge (generic function with 1 method)

### Run sparse ridge traces and plot

Loops over `RIDGE_SPARSE_CONFIG`, concatenates all ridge hits into `ridge_x`, `ridge_y`, `ridge_z`, and overlays them on the coil geometry (mirrors the end of `Magnetic_Ridge.jl`).

In [ ]:
ridge_x, ridge_y, ridge_z = Float64[], Float64[], Float64[]
poincare_x, poincare_y, poincare_z = Float64[], Float64[], Float64[]

for (phi, (ra, rb, nr)) in RIDGE_SPARSE_CONFIG
    for r in range(ra, rb; length = nr)
        println("Tracing ridge probe: φ = ", round(phi / π; digits = 3), "π, r = ", round(r; digits = 4))
        px, py, pz, rx, ry, rz = run_poincare_ridge(coils, r, phi, L_RIDGE_TRACE)
        append!(poincare_x, px)
        append!(poincare_y, py)
        append!(poincare_z, pz)
        append!(ridge_x, rx)
        append!(ridge_y, ry)
        append!(ridge_z, rz)
    end
end

println("Sparse ridge points collected: ", length(ridge_z))

fig = Figure(size = (1200, 900))
ax = Axis3(fig[1, 1]; title = "Coils + Poincaré (red) + sparse ridge peaks (purple)", aspect = :data)
for c in coils
    pts = c.rs
    xs = [p[1] for p in pts]
    ys = [p[2] for p in pts]
    zs = [p[3] for p in pts]
    push!(xs, xs[1])
    push!(ys, ys[1])
    push!(zs, zs[1])
    lines!(ax, xs, ys, zs; color = :steelblue, linewidth = 2, alpha = 0.55)
end
scatter!(ax, poincare_x, poincare_y, poincare_z; color = :red, markersize = 3)
scatter!(ax, ridge_x, ridge_y, ridge_z; color = :purple, markersize = 5)
fig

### XY footprint for dense scan (`magnetic_ridge_3.jl` — setup)

The dense stage needs:

1. A **footprint** in XY so we do not waste roots outside the confinement region; here the footprint is built from the **sparse ridge cloud** you just produced (same angular binning as `build_xy_footprint` in `magnetic_ridge_3.jl`).
2. A **local $z$ interval** per column $(x,y)$ estimated from neighboring ridge points (hash-grid + radius growth), again matching `magnetic_ridge_3.jl`.

Tune grid counts and padding constants before running the next cells. Start modest (`NX`, `NY` $\approx 80$) to verify behavior, then increase resolution.

In [ ]:
# --- ForwardDiff ridge helpers (from magnetic_ridge_3.jl) -----------------------
function B_agnostic(cs::Vector{Coil{T}}, x::AbstractVector) where {T}
    B_acc = StaticArrays.zeros(eltype(x), 3)
    for c in cs
        for (ri, Idri) in zip(c.rs, c.Idrs)
            delta_r = SVector(ri[1] - x[1], ri[2] - x[2], ri[3] - x[3])
            d = three_norm(delta_r)
            B_acc = B_acc + cross(delta_r, Idri) / (d^3)
        end
    end
    return B_acc
end

function d_normB_ds(cs::Vector{Coil{T}}, x, y, z) where {T}
    u = SVector{3}(x, y, z)
    B = eval_B(cs, u)
    J = ForwardDiff.jacobian(v -> B_agnostic(cs, v), u)
    return dot(B, J, B)
end

const RIDGE_CURV_EPS = 1e-5
const B_MAG_MAX = 10.0

function is_ridge(cs::Vector{Coil{T}}, x, y, z) where {T}
    u = SVector{3}(x, y, z)
    B = eval_B(cs, u)
    nb = norm(B)
    nb < eps(Float64) * 1e6 && return false
    bh = B / nb
    epss = RIDGE_CURV_EPS
    s0 = d_normB_ds(cs, x, y, z)
    s1 = d_normB_ds(cs, x + epss * bh[1], y + epss * bh[2], z + epss * bh[3])
    return ((s1 - s0) / epss) < 0.0
end

# Spatial index + footprint structs/functions (abbreviated from magnetic_ridge_3.jl)
struct XYIndex
    x::Vector{Float64}
    y::Vector{Float64}
    z::Vector{Float64}
    h::Float64
    invh::Float64
    bins::Dict{Tuple{Int,Int},Vector{Int}}
end

@inline cell_id(x, y, invh) = (floor(Int, x * invh), floor(Int, y * invh))

function build_xy_index(cx::Vector{Float64}, cy::Vector{Float64}, cz::Vector{Float64}; h::Float64 = 0.25)
    bins = Dict{Tuple{Int,Int},Vector{Int}}()
    invh = 1.0 / h
    @inbounds for i in eachindex(cx)
        key = cell_id(cx[i], cy[i], invh)
        if haskey(bins, key)
            push!(bins[key], i)
        else
            bins[key] = [i]
        end
    end
    return XYIndex(cx, cy, cz, h, invh, bins)
end

function gather_within_radius(idx::XYIndex, x::Float64, y::Float64, r::Float64)
    ii, jj = cell_id(x, y, idx.invh)
    rr = ceil(Int, r / idx.h)
    r2 = r * r
    out = Int[]
    for i in (ii-rr):(ii+rr), j in (jj-rr):(jj+rr)
        ids = get(idx.bins, (i, j), nothing)
        ids === nothing && continue
        @inbounds for k in ids
            dx = idx.x[k] - x
            dy = idx.y[k] - y
            if dx * dx + dy * dy <= r2
                push!(out, k)
            end
        end
    end
    return out
end

function local_confine_bounds(
    idx::XYIndex,
    x::Float64,
    y::Float64,
    dx::Float64,
    dy::Float64;
    xy_radius_base::Float64 = 0.06,
    xy_radius_max::Float64 = 0.25,
    min_neighbors::Int = 18,
    z_pad::Float64 = 0.01,
)
    r = max(xy_radius_base, 0.75 * hypot(dx, dy))
    ids = gather_within_radius(idx, x, y, r)
    while length(ids) < min_neighbors && r < xy_radius_max
        r = min(xy_radius_max, 1.55 * r)
        ids = gather_within_radius(idx, x, y, r)
    end
    isempty(ids) && return nothing
    zmin = Inf
    zmax = -Inf
    @inbounds for k in ids
        zk = idx.z[k]
        zk < zmin && (zmin = zk)
        zk > zmax && (zmax = zk)
    end
    return (zmin - z_pad, zmax + z_pad, length(ids), r)
end

struct XYFootprint
    ntheta::Int
    rmin::Vector{Float64}
    rmax::Vector{Float64}
end

@inline function theta_bin(theta::Float64, ntheta::Int)
    t = (theta + π) / (2π)
    return clamp(floor(Int, t * ntheta) + 1, 1, ntheta)
end

function _fill_circular_nearest!(v::Vector{Float64})
    n = length(v)
    finite_idx = findall(isfinite, v)
    isempty(finite_idx) && error("Footprint bins are empty; supply more ridge points.")
    for i in eachindex(v)
        isfinite(v[i]) && continue
        best_j = finite_idx[1]
        best_d = min(abs(i - best_j), n - abs(i - best_j))
        for j in finite_idx
            d = min(abs(i - j), n - abs(i - j))
            if d < best_d
                best_d = d
                best_j = j
            end
        end
        v[i] = v[best_j]
    end
    return v
end

function build_xy_footprint(cx::Vector{Float64}, cy::Vector{Float64}; ntheta::Int = 180)
    rmin = fill(Inf, ntheta)
    rmax = fill(-Inf, ntheta)
    @inbounds for i in eachindex(cx)
        x = cx[i]
        y = cy[i]
        r = hypot(x, y)
        b = theta_bin(atan(y, x), ntheta)
        r < rmin[b] && (rmin[b] = r)
        r > rmax[b] && (rmax[b] = r)
    end
    _fill_circular_nearest!(rmin)
    _fill_circular_nearest!(rmax)
    rmin_s = similar(rmin)
    rmax_s = similar(rmax)
    for i in 1:ntheta
        i0 = mod1(i - 1, ntheta)
        i1 = i
        i2 = mod1(i + 1, ntheta)
        rmin_s[i] = 0.25 * rmin[i0] + 0.5 * rmin[i1] + 0.25 * rmin[i2]
        rmax_s[i] = 0.25 * rmax[i0] + 0.5 * rmax[i1] + 0.25 * rmax[i2]
    end
    return XYFootprint(ntheta, rmin_s, rmax_s)
end

@inline function in_xy_footprint(fp::XYFootprint, x::Float64, y::Float64; inner_pad::Float64 = 0.04, outer_pad::Float64 = 0.07)
    r = hypot(x, y)
    b = theta_bin(atan(y, x), fp.ntheta)
    rlo = max(0.0, fp.rmin[b] - inner_pad)
    rhi = fp.rmax[b] + outer_pad
    return (r >= rlo) && (r <= rhi)
end

### Define XY grid and footprint mask (run after sparse ridge data exist)

This builds `idx` from `(ridge_x, ridge_y, ridge_z)` and constructs `fp` for masking. A diagnostic figure shows whether the angular annulus covers your sparse points; tighten or loosen `inner_pad` / `outer_pad` in the next stage if needed.

In [ ]:
# Cloud used as surrogate "confinement" samples (your sparse ridge extraction)
conf_x = copy(ridge_x)
conf_y = copy(ridge_y)
conf_z = copy(ridge_z)

const FOOTPRINT_NTHETA = 180
const XY_BOX_PAD = 0.03
const INDEX_CELL_H = 0.25  # hash grid cell size for neighbor queries

idx = build_xy_index(conf_x, conf_y, conf_z; h = INDEX_CELL_H)
fp = build_xy_footprint(conf_x, conf_y; ntheta = FOOTPRINT_NTHETA)

x_min = minimum(conf_x) - XY_BOX_PAD
x_max = maximum(conf_x) + XY_BOX_PAD
y_min = minimum(conf_y) - XY_BOX_PAD
y_max = maximum(conf_y) + XY_BOX_PAD

println("Bounding box [x] × [y]: ", (x_min, x_max), " × ", (y_min, y_max))

# Quick footprint visualization
figm = Figure(size = (700, 650))
axm = Axis(figm[1, 1]; title = "Sparse ridge footprint (inner/outer polar bounds)", aspect = DataAspect())
scatter!(axm, conf_x, conf_y; markersize = 4, color = (:black, 0.35))
th = range(-π, π; length = fp.ntheta + 1)
xo, yo = Float64[], Float64[]
xi, yi = Float64[], Float64[]
inner_pad, outer_pad = 0.04, 0.07
for t in th
    b = theta_bin(t, fp.ntheta)
    rlo = max(0.0, fp.rmin[b] - inner_pad)
    rhi = fp.rmax[b] + outer_pad
    push!(xo, rhi * cos(t))
    push!(yo, rhi * sin(t))
    push!(xi, rlo * cos(t))
    push!(yi, rlo * sin(t))
end
lines!(axm, xo, yo; color = :dodgerblue, linewidth = 2)
lines!(axm, xi, yi; color = :tomato, linewidth = 2)
figm

### Dense ridge scan: root finding with ForwardDiff

For each XY node inside the footprint we:

1. Estimate a bracket $[z_\mathrm{lo}, z_\mathrm{hi}]$ from neighboring sparse ridge samples;
2. Solve $d|\mathbf{B}|/ds = 0$ with `find_zeros` on that interval;
3. Filter roots with $|{\bf B}| \le$ `B_MAG_MAX` and the curvature test `is_ridge`.

If multiple roots qualify, we keep the one with largest $|{\bf B}|$ (same tie-break as `magnetic_ridge_3.jl`). Adjust `NX`, `NY` here.

In [ ]:
const NX = 120
const NY = 120

function map_ridge_surface_dense(
    cs::Vector{Coil{T}},
    idx::XYIndex,
    fp::XYFootprint,
    x_min::Float64,
    x_max::Float64,
    y_min::Float64,
    y_max::Float64;
    nx::Int = NX,
    ny::Int = NY,
    inner_pad::Float64 = 0.04,
    outer_pad::Float64 = 0.07,
) where {T}
    xs = collect(range(x_min, x_max; length = nx))
    ys = collect(range(y_min, y_max; length = ny))
    dx = (x_max - x_min) / max(nx - 1, 1)
    dy = (y_max - y_min) / max(ny - 1, 1)

    dridge_x = Float64[]
    dridge_y = Float64[]
    dridge_z = Float64[]

    grid_x = Float64[]
    grid_y = Float64[]
    grid_z = Float64[]

    for x in xs
        for y in ys
            push!(grid_x, x)
            push!(grid_y, y)
            if !in_xy_footprint(fp, x, y; inner_pad = inner_pad, outer_pad = outer_pad)
                push!(grid_z, NaN)
                continue
            end
            bounds = local_confine_bounds(idx, x, y, dx, dy)
            if bounds === nothing
                push!(grid_z, NaN)
                continue
            end
            z_lo, z_hi = bounds[1], bounds[2]
            z_lo >= z_hi && (push!(grid_z, NaN); continue)

            fz(z) = d_normB_ds(cs, x, y, z)
            roots = try
                find_zeros(fz, z_lo, z_hi)
            catch
                Float64[]
            end

            valid_z = Float64[]
            valid_B = Float64[]
            for z in roots
                Bmag = norm(eval_B(cs, SVector{3}(x, y, z)))
                Bmag > B_MAG_MAX && continue
                (z < z_lo || z > z_hi) && continue
                is_ridge(cs, x, y, z) || continue
                push!(valid_z, z)
                push!(valid_B, Bmag)
            end

            if isempty(valid_z)
                push!(grid_z, NaN)
                continue
            end
            k = argmax(valid_B)
            z_pick = valid_z[k]
            push!(grid_z, z_pick)
            push!(dridge_x, x)
            push!(dridge_y, y)
            push!(dridge_z, z_pick)
        end
    end
    return dridge_x, dridge_y, dridge_z, grid_x, grid_y, grid_z
end

### Execute dense scan and plot the refined ridge cloud

This cell performs the heavy numerical work. When satisfied, use the following cell to persist a JLD2 bundle for `Magnetic_Ridge_Set_2.ipynb`.

In [ ]:
drx, dry, drz, gx, gy, gz = map_ridge_surface_dense(
    coils,
    idx,
    fp,
    x_min,
    x_max,
    y_min,
    y_max;
    nx = NX,
    ny = NY,
)

println("Dense valid ridge points: ", length(drz))

figd = Figure(size = (1100, 850))
axd = Axis3(
    figd[1, 1];
    title = "Dense magnetic ridge peaks (ForwardDiff + root search)",
    aspect = :data,
)
for c in coils
    pts = c.rs
    xs = [p[1] for p in pts]
    ys = [p[2] for p in pts]
    zs = [p[3] for p in pts]
    push!(xs, xs[1])
    push!(ys, ys[1])
    push!(zs, zs[1])
    lines!(axd, xs, ys, zs; color = (:steelblue, 0.45), linewidth = 1.5)
end
Rc = sqrt.(drx .^ 2 .+ dry .^ 2)
scatter!(axd, drx, dry, drz; markersize = 5, color = Rc, colormap = :plasma)
figd

### Optional export (JLD2) for Set 2

Writes `ridge_x`, `ridge_y`, `ridge_z` (dense valid samples), optional full `grid_*` arrays with `NaN` masks, and lightweight metadata. Adjust the destination path if desired.

In [ ]:
out_dir = joinpath(REPO, "data", "magnetic_ridge3")
mkpath(out_dir)
out_jld2 = joinpath(out_dir, "magnetic_ridge_set1_dense.jld2")

JLD2.jldopen(out_jld2, "w") do f
    f["ridge_x"] = drx
    f["ridge_y"] = dry
    f["ridge_z"] = drz
    f["grid_x"] = gx
    f["grid_y"] = gy
    f["grid_z"] = gz
    f["sparse_ridge_x"] = ridge_x
    f["sparse_ridge_y"] = ridge_y
    f["sparse_ridge_z"] = ridge_z
    f["meta"] = Dict(
        "source_note" => "Magnetic_Ridge_Set_1.ipynb dense stage; footprint from sparse ridge trace peaks",
        "coil_file" => COIL_FILE,
        "nx" => NX,
        "ny" => NY,
        "l_poincare" => L_POINCARE,
        "l_ridge_trace" => L_RIDGE_TRACE,
    )
end

println("Saved ", out_jld2)